In [10]:

from pathlib import Path
import pandas as pd
import numpy as np

print("Libraries imported successfully.")


Libraries imported successfully.


In [11]:

file_path = Path("data/raw/Online Retail.xlsx")

if not file_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {file_path.resolve()}"
    )

if file_path.suffix.lower() != ".xlsx":
    raise ValueError("Expected an Excel .xlsx file.")

df = pd.read_excel(file_path)

print("Dataset loaded successfully.")
print("File format:", file_path.suffix)
print("Dataset shape:", df.shape)



Dataset loaded successfully.
File format: .xlsx
Dataset shape: (541909, 8)


In [12]:

expected_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]

missing_columns = set(expected_columns) - set(df.columns)
unexpected_columns = set(df.columns) - set(expected_columns)

print("Expected columns:", expected_columns)
print("Missing expected columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

assert len(missing_columns) == 0, "Required columns are missing!"
assert len(unexpected_columns) == 0, "Unexpected columns found!"

print("Schema validation passed.")


Expected columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Missing expected columns: set()
Unexpected columns: set()
Schema validation passed.


In [13]:

schema_summary = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Null_Count": df.notna().sum().values,
    "Missing_Count": df.isna().sum().values
})

display(schema_summary)


,Column,Data_Type,Non_Null_Count,Missing_Count
0,InvoiceNo,object,541909,0
1,StockCode,object,541909,0
2,Description,object,540455,1454
3,Quantity,int64,541909,0
4,InvoiceDate,datetime64[us],541909,0
5,UnitPrice,float64,541909,0
6,CustomerID,float64,406829,135080
7,Country,str,541909,0


In [14]:

print("First 10 records:")
display(df.head(10))

print("Last 5 records:")
display(df.tail())


First 10 records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


Last 5 records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [15]:

print("Missing values per column:")
display(df.isnull().sum().to_frame("Missing_Count"))

print("Exact duplicate rows:", df.duplicated().sum())

print("\nDataset information:")
df.info()


Missing values per column:


,Missing_Count
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


Exact duplicate rows: 5268

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [16]:

retail = df.copy()

# Remove exact duplicate rows
retail = retail.drop_duplicates()

# Remove records without a product description
retail = retail.dropna(subset=["Description"])

# Keep positive prices for initial sales analysis
retail = retail[retail["UnitPrice"] > 0]

# Keep positive quantities for initial sales analysis
retail = retail[retail["Quantity"] > 0]

# Calculate line-item revenue
retail["Revenue"] = retail["Quantity"] * retail["UnitPrice"]

print("Original shape:", df.shape)
print("Preliminary analysis shape:", retail.shape)

print("\nRemaining missing values:")
display(retail.isnull().sum().to_frame("Missing_Count"))

display(retail.head())


Original shape: (541909, 8)
Preliminary analysis shape: (524878, 9)

Remaining missing values:


,Missing_Count
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,132186
Country,0
Revenue,0


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [17]:

display(
    retail[["Quantity", "UnitPrice", "Revenue"]]
    .describe()
    .round(2)
)


,Quantity,UnitPrice,Revenue
count,524878.00,524878.00,524878.00
mean,10.62,3.92,20.28
std,156.28,36.09,271.69
min,1.00,0.00,0.00
25%,1.00,1.25,3.90
50%,4.00,2.08,9.92
75%,11.00,4.13,17.70
max,80995.00,13541.33,168469.60


In [18]:

product_revenue = (
    retail.groupby(["StockCode", "Description"], as_index=False)
    .agg(
        Total_Quantity=("Quantity", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Average_Price=("UnitPrice", "mean"),
        Transaction_Count=("InvoiceNo", "nunique")
    )
    .sort_values("Total_Revenue", ascending=False)
)

print("Top 10 product entries by revenue:")
display(product_revenue.head(10))


Top 10 product entries by revenue:


,StockCode,Description,Total_Quantity,Total_Revenue,Average_Price,Transaction_Count
4150,DOT,DOTCOM POSTAGE,706,206248.77,292.137068,706
1262,22423,REGENCY CAKESTAND 3 TIER,13851,174156.54,13.983936,1988
2590,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,2.080000,1
3773,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37580,104284.24,3.116159,2189
2669,47566,PARTY BUNTING,18283,99445.23,5.797928,1685
3761,85099B,JUMBO BAG RED RETROSPOT,48371,94159.81,2.486197,2089
2045,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92,1.468480,247
4153,POST,POSTAGE,3150,78101.88,31.076581,1126
4151,M,Manual,6984,77750.27,234.489652,289
1951,23084,RABBIT NIGHT LIGHT,30739,66870.03,2.386401,994


In [19]:

country_revenue = (
    retail.groupby("Country", as_index=False)
    .agg(
        Total_Revenue=("Revenue", "sum"),
        Total_Quantity=("Quantity", "sum"),
        Invoice_Count=("InvoiceNo", "nunique")
    )
    .sort_values("Total_Revenue", ascending=False)
)

print("Top 10 countries by revenue:")
display(country_revenue.head(10))


Top 10 countries by revenue:


,Country,Total_Revenue,Total_Quantity,Invoice_Count
36,United Kingdom,9001744.094,4646906,18019
24,Netherlands,285446.340,200361,94
10,EIRE,283140.520,147007,288
14,Germany,228678.400,119154,457
13,France,209625.370,112060,392
0,Australia,138453.810,83891,57
31,Spain,61558.560,27933,90
33,Switzerland,57067.600,30617,54
3,Belgium,41196.340,23237,98
32,Sweden,38367.830,36078,36


In [20]:

print("=" * 45)
print("DAY 2 — DATASET DECISION SUMMARY")
print("=" * 45)

print("Selected dataset: UCI Online Retail")
print("File format: Excel (.xlsx)")
print("Original rows:", df.shape[0])
print("Original columns:", df.shape[1])
print("Analysis rows:", retail.shape[0])
print("Analysis columns:", retail.shape[1])
print("Original duplicate rows:", df.duplicated().sum())
print("Missing CustomerID values:", df["CustomerID"].isna().sum())

print("\nDeliverable status:")
print("[x] Dataset loaded")
print("[x] Schema inspected")
print("[x] Sample preview generated")
print("[x] Missing values and duplicates inspected")
print("[x] Preliminary revenue analysis completed")
print("[ ] Dataset decision document saved separately")


DAY 2 — DATASET DECISION SUMMARY
Selected dataset: UCI Online Retail
File format: Excel (.xlsx)
Original rows: 541909
Original columns: 8
Analysis rows: 524878
Analysis columns: 9
Original duplicate rows: 5268
Missing CustomerID values: 135080

Deliverable status:
[x] Dataset loaded
[x] Schema inspected
[x] Sample preview generated
[x] Missing values and duplicates inspected
[x] Preliminary revenue analysis completed
[ ] Dataset decision document saved separately
